In [12]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

In [68]:
def generate_quantum_random_bits(num_bits):
    """
    Generates a list of random bits (0 or 1) using quantum measurement.
    Each bit is generated by measuring a |+> state.
    """
    random_bits = []
    for _ in range(num_bits):
        qc = QuantumCircuit(1, 1) # 1 qubit, 1 classical bit
        qc.h(0) # Apply Hadamard to create |+> state
        qc.measure(0, 0) # Measure in the computational basis

        simulator = BasicSimulator()
        compiled_circuit = transpile(qc, simulator)
        job = simulator.run(compiled_circuit, shots=1)
        result = job.result()
        counts = result.get_counts(qc)

        # The result will be either {'0': 1} or {'1': 1}
        if '0' in counts: # If '0' was measured
            random_bits.append(0)
        else:
            random_bits.append(1)

    return random_bits

# Example usage (can be removed later if not needed for direct display)
# num_bits_for_example = 10
# random_q_bits = generate_quantum_random_bits(num_bits_for_example)
# print(f"Generated {num_bits_for_example} quantum random bits: {random_q_bits}")

In [70]:
KEY_LENGTH = 10 # This defines how many bits Alice will try to send. For a demo, a small number like 10 is good.

print("--- ALICE'S PART: PREPARING AND SENDING QUBITS ---")

# Step 1: Alice generates her secret key bits (these are the '0's and '1's she wants to share)
alice_key_bits = generate_quantum_random_bits(KEY_LENGTH)
print(f"1. Alice's secret key bits (the random sequence she wants to establish with Bob): {alice_key_bits}")

# Step 2: Alice chooses a random basis for each bit.
# 0 = Z basis (computational basis, {|0>, |1>})
# 1 = X basis (Hadamard basis, {|+>, |->})
alice_bases = generate_quantum_random_bits(KEY_LENGTH)
print(f"2. Alice's randomly chosen bases for encoding each bit: {alice_bases} (0=Z basis, 1=X basis)")

# Step 3: Alice encodes each of her secret bits into a quantum bit (qubit) based on her chosen basis.
# If bit is 0 in Z basis: she sends |0>
# If bit is 1 in Z basis: she sends |1> (by applying an X gate to |0>)
# If bit is 0 in X basis: she sends |+> (by applying an H gate to |0>)
# If bit is 1 in X basis: she sends |-> (by applying an H gate and then X gate to |0>)
quantum_channel_qubits = [] # This list will hold the prepared qubits that Alice sends
for i in range(KEY_LENGTH):
    qc = QuantumCircuit(1, 1) # Create a new quantum circuit for each qubit (1 qubit, 1 classical bit)
    bit = alice_key_bits[i]
    basis = alice_bases[i]

    if bit == 1: # If the secret bit is 1, apply an X gate to flip |0> to |1>
        qc.x(0)

    if basis == 1: # If Alice chose the X basis, apply a Hadamard gate
        qc.h(0)

    quantum_channel_qubits.append(qc) # Add the prepared qubit circuit to the list that represents the quantum channel

print(f"3. Alice has prepared {len(quantum_channel_qubits)} qubits, encoding her bits in her chosen bases, and sent them through the quantum channel.")


--- ALICE'S PART: PREPARING AND SENDING QUBITS ---
1. Alice's secret key bits (the random sequence she wants to establish with Bob): [0, 0, 1, 1, 1, 0, 0, 0, 0, 0]
2. Alice's randomly chosen bases for encoding each bit: [0, 1, 1, 0, 1, 0, 0, 1, 0, 0] (0=Z basis, 1=X basis)
3. Alice has prepared 10 qubits, encoding her bits in her chosen bases, and sent them through the quantum channel.


In [71]:
print("\n--- BOB'S PART: RECEIVING AND MEASURING QUBITS ---")

# Step 4: Bob doesn't know Alice's bases, so he randomly chooses his own basis for measuring each incoming qubit.
bob_bases = generate_quantum_random_bits(KEY_LENGTH)
print(f"4. Bob's randomly chosen bases for measurement: {bob_bases} (0=Z basis, 1=X basis)")

# Step 5: Bob measures each received qubit using his randomly chosen basis.
# If Bob's basis matches Alice's, he has a high chance of getting the correct bit.
# If Bob's basis doesn't match Alice's, his measurement result will be random.
bob_measured_bits = [] # This list will store the bits Bob measures
simulator = BasicSimulator()

for i in range(KEY_LENGTH):
    qc_to_measure = quantum_channel_qubits[i] # Get the qubit circuit Alice sent
    bob_basis = bob_bases[i]

    # Create a copy of Alice's circuit to add Bob's measurement operations without altering Alice's original state setup
    bob_qc = qc_to_measure.copy_empty_like()
    for instruction in qc_to_measure.data: # Reconstruct Alice's operations to ensure Bob operates on her prepared state
        bob_qc.append(instruction)

    if bob_basis == 1: # If Bob chose the X basis, apply a Hadamard gate before measuring (to measure in X basis)
        bob_qc.h(0)

    bob_qc.measure(0, 0) # Measure the qubit in the standard computational (Z) basis

    # Simulate the quantum circuit to get the measurement result
    compiled_circuit = transpile(bob_qc, simulator)
    job = simulator.run(compiled_circuit, shots=1) # Run the circuit once to get one measurement
    result = job.result()
    counts = result.get_counts(bob_qc) # Get the measurement outcomes (e.g., {'0': 1} or {'1': 1})

    # Extract the measured bit (0 or 1)
    if '0' in counts:
        bob_measured_bits.append(0)
    else:
        bob_measured_bits.append(1)

print(f"5. Bob has measured all {KEY_LENGTH} received qubits using his random bases.")
print(f"   Bob's measured bits: {bob_measured_bits}")


--- BOB'S PART: RECEIVING AND MEASURING QUBITS ---
4. Bob's randomly chosen bases for measurement: [0, 1, 1, 0, 1, 0, 1, 1, 0, 1] (0=Z basis, 1=X basis)
5. Bob has measured all 10 received qubits using his random bases.
   Bob's measured bits: [0, 0, 1, 1, 1, 0, 0, 0, 0, 1]


In [72]:
print("\n--- CLASSICAL COMMUNICATION: BASIS SIFTING (Public Announcement) ---")
alice_sifted_key = [] # This will store Alice's key bits where bases matched
bob_sifted_key = []   # This will store Bob's measured bits where bases matched

print("6. Alice and Bob publicly announce *only* their chosen bases for each qubit (not the bit values themselves).")
print("   They then compare these announced bases to find positions where they used the same measurement setting.")

print("\n   Detailed Basis Comparison:")
for i in range(KEY_LENGTH):
    alice_basis = alice_bases[i]
    bob_basis = bob_bases[i]
    match_status = "MATCHED" if alice_basis == bob_basis else "DID NOT MATCH"
    print(f"   Position {i+1}: Alice's Basis = {alice_basis} (0=Z, 1=X), Bob's Basis = {bob_basis} (0=Z, 1=X) -> {match_status}")

    # They keep only the bits where their chosen bases were the same.
    if alice_basis == bob_basis:
        alice_sifted_key.append(alice_key_bits[i])     # Alice adds her original bit to her sifted key
        bob_sifted_key.append(bob_measured_bits[i])   # Bob adds his measured bit to his sifted key

print(f"\n   Number of positions where Alice's and Bob's bases matched: {len(alice_sifted_key)} out of {KEY_LENGTH}")
print(f"   Alice's raw shared key (based on matching bases): {alice_sifted_key}")
print(f"   Bob's raw shared key (based on matching bases): {bob_sifted_key}")


--- CLASSICAL COMMUNICATION: BASIS SIFTING (Public Announcement) ---
6. Alice and Bob publicly announce *only* their chosen bases for each qubit (not the bit values themselves).
   They then compare these announced bases to find positions where they used the same measurement setting.

   Detailed Basis Comparison:
   Position 1: Alice's Basis = 0 (0=Z, 1=X), Bob's Basis = 0 (0=Z, 1=X) -> MATCHED
   Position 2: Alice's Basis = 1 (0=Z, 1=X), Bob's Basis = 1 (0=Z, 1=X) -> MATCHED
   Position 3: Alice's Basis = 1 (0=Z, 1=X), Bob's Basis = 1 (0=Z, 1=X) -> MATCHED
   Position 4: Alice's Basis = 0 (0=Z, 1=X), Bob's Basis = 0 (0=Z, 1=X) -> MATCHED
   Position 5: Alice's Basis = 1 (0=Z, 1=X), Bob's Basis = 1 (0=Z, 1=X) -> MATCHED
   Position 6: Alice's Basis = 0 (0=Z, 1=X), Bob's Basis = 0 (0=Z, 1=X) -> MATCHED
   Position 7: Alice's Basis = 0 (0=Z, 1=X), Bob's Basis = 1 (0=Z, 1=X) -> DID NOT MATCH
   Position 8: Alice's Basis = 1 (0=Z, 1=X), Bob's Basis = 1 (0=Z, 1=X) -> MATCHED
   Position 9

In [73]:
print("\n--- CLASSICAL COMMUNICATION: ERROR CHECKING (Verifying the Key) ---")

# Step 7: Alice and Bob compare a small, randomly chosen subset of their sifted keys.
# In a real scenario, they would sacrifice some key bits to check for errors. Here, we compare the full sifted key.
error_count_in_sifted_key = 0
mismatch_indices_in_sifted_key = [] # To store the positions *within the sifted key* where errors occurred

print(f"7. Alice and Bob examine the outcome for each of the original {KEY_LENGTH} positions, focusing on the sifted key for error detection.")
print("\n   Detailed Key Verification (across all original positions):")

sifted_index_counter = 0 # To track the index within the actual sifted keys

for i in range(KEY_LENGTH):
    alice_basis = alice_bases[i]
    bob_basis = bob_bases[i]

    if alice_basis == bob_basis:
        # This position was part of the sifted key, perform error check
        alice_sifted_bit = alice_sifted_key[sifted_index_counter] # Get the bit from the already formed sifted key
        bob_sifted_bit = bob_sifted_key[sifted_index_counter]     # Get the bit from the already formed sifted key

        match_status = "MATCH" if alice_sifted_bit == bob_sifted_bit else "MISMATCH!"
        print(f"   Original Position {i+1}: Bases MATCHED. Alice's Bit (sifted) = {alice_sifted_bit}, Bob's Bit (sifted) = {bob_sifted_bit} -> {match_status}")

        if alice_sifted_bit != bob_sifted_bit:
            error_count_in_sifted_key += 1
            mismatch_indices_in_sifted_key.append(sifted_index_counter)
        sifted_index_counter += 1 # Increment for the next sifted bit
    else:
        # This position was discarded
        print(f"   Original Position {i+1}: Bases DID NOT MATCH. This position was DISCARDED because Bob's measurement was random.")

print(f"\n   Total number of mismatches found in the SIFTED key: {error_count_in_sifted_key}")

if error_count_in_sifted_key == 0:
    print(f"   Result: Alice and Bob successfully established a shared secret key with no errors!")
    print(f"   Since there are no errors, they can be confident no eavesdropper (Eve) was present.")
    print(f"   The shared secret key (from {len(alice_sifted_key)} sifted bits) is: {alice_sifted_key}")
else:
    print(f"   Result: Errors detected in the SIFTED key! There were {error_count_in_sifted_key} mismatches out of {len(alice_sifted_key)} sifted bits.")
    print(f"   This means the error rate is {error_count_in_sifted_key / len(alice_sifted_key):.2%}.")
    print("   In a real BB84 protocol, this would indicate either channel noise or the presence of an eavesdropper.")
    print("   They would then discard this key and restart the process.")
    if mismatch_indices_in_sifted_key:
        first_mismatch_sifted_idx = mismatch_indices_in_sifted_key[0]
        print(f"   Example mismatch at sifted key index {first_mismatch_sifted_idx} (Alice's bit: {alice_sifted_key[first_mismatch_sifted_idx]}, Bob's measured bit: {bob_sifted_key[first_mismatch_sifted_idx]})")


--- CLASSICAL COMMUNICATION: ERROR CHECKING (Verifying the Key) ---
7. Alice and Bob examine the outcome for each of the original 10 positions, focusing on the sifted key for error detection.

   Detailed Key Verification (across all original positions):
   Original Position 1: Bases MATCHED. Alice's Bit (sifted) = 0, Bob's Bit (sifted) = 0 -> MATCH
   Original Position 2: Bases MATCHED. Alice's Bit (sifted) = 0, Bob's Bit (sifted) = 0 -> MATCH
   Original Position 3: Bases MATCHED. Alice's Bit (sifted) = 1, Bob's Bit (sifted) = 1 -> MATCH
   Original Position 4: Bases MATCHED. Alice's Bit (sifted) = 1, Bob's Bit (sifted) = 1 -> MATCH
   Original Position 5: Bases MATCHED. Alice's Bit (sifted) = 1, Bob's Bit (sifted) = 1 -> MATCH
   Original Position 6: Bases MATCHED. Alice's Bit (sifted) = 0, Bob's Bit (sifted) = 0 -> MATCH
   Original Position 7: Bases DID NOT MATCH. This position was DISCARDED because Bob's measurement was random.
   Original Position 8: Bases MATCHED. Alice's Bit 

In [74]:
print("--- BB84 KEY ESTABLISHMENT SUMMARY ---")

# After basis sifting and error checking, alice_sifted_key and bob_sifted_key should be identical.
# This is the shared secret key established by the BB84 protocol.

print(f"BB84 initial qubit count (KEY_LENGTH): {KEY_LENGTH}")
print(f"Alice's BB84 raw key bits (used to encode qubits): {alice_key_bits}")
print(f"Successfully established shared secret key (Alice's sifted key): {alice_sifted_key}")
print(f"Length of shared secret key: {len(alice_sifted_key)}")


--- BB84 KEY ESTABLISHMENT SUMMARY ---
BB84 initial qubit count (KEY_LENGTH): 10
Alice's BB84 raw key bits (used to encode qubits): [0, 0, 1, 1, 1, 0, 0, 0, 0, 0]
Successfully established shared secret key (Alice's sifted key): [0, 0, 1, 1, 1, 0, 0, 0]
Length of shared secret key: 8
